# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafiz-Taha-Hussain/Flyrank-Work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — connect to the warehouse (run once)

Same pattern as w03: authenticate, then build the March feature frame we'll build the rule
on. April is used ONLY to verify signals below (never as a rule input) — see the leakage
check in section 4 for the explicit proof of that.

In [1]:
!pip install -q duckdb huggingface_hub

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import login

from google.colab import userdata
HF_TOKEN = userdata.get("HF-TOKEN")  # match whatever you actually named the Colab secret
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');""")

MARCH_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-03/*.parquet"
APRIL_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-04/*.parquet"

features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS avg_position_march,
    SUM(gsc_impressions)                                            AS impressions_march,
    SUM(gsc_clicks)                                                 AS clicks_march,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_march
FROM '{MARCH_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 6)


,client_hash_id,content_hash_id,avg_position_march,impressions_march,clicks_march,ctr_march
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,5.171271,181.0,0.0,0.000000
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,4.543478,46.0,1.0,0.021739
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,5.825362,899.0,1.0,0.001112
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,5.941176,34.0,0.0,0.000000
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,6.953668,3108.0,0.0,0.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth reviewing if it still gets real search
demand, but its CTR is meaningfully below what other pages ranking at the same position
typically get — i.e. it's ranking fine but losing clicks it "should" be getting for that
position. That's the CTR-fix pattern from the session, applied to my lane.

**Reason code:** `ctr_underperforming_for_position_tier` (one reason code, as the skill
asks — the rule either fires with this reason or it doesn't fire at all).

**Action label:** `review_for_ctr_fix`.

Before encoding it, I'm checking two signals the rule leans on — one is the CTR-vs-position
relationship itself (does CTR really vary by position the way the rule assumes?), the other
is whether search volume genuinely relates to real outcomes (the quick-win premise: a
declining page with real traffic matters more than a declining page nobody sees).

### Signal check 1 — CTR vs. position (behind the CTR-fix flag)

If CTR doesn't actually track position, then "low CTR for this position tier" isn't a real
signal — it's noise. Bucketing March data by position tier and printing `n` + mean CTR per
bucket.

In [2]:
def position_tier(pos):
    if pd.isna(pos):
        return "no_data"
    elif pos <= 3:
        return "1-3"
    elif pos <= 10:
        return "4-10"
    elif pos <= 20:
        return "11-20"
    elif pos <= 50:
        return "21-50"
    else:
        return "51+"

features["position_tier"] = features["avg_position_march"].apply(position_tier)

tier_order = ["1-3", "4-10", "11-20", "21-50", "51+", "no_data"]
signal1_table = features.groupby("position_tier").agg(
    n=("content_hash_id", "size"),
    mean_ctr=("ctr_march", "mean"),
    median_ctr=("ctr_march", "median"),
).reindex(tier_order)
signal1_table

,n,mean_ctr,median_ctr
position_tier,,,
1-3,18860,0.011696,0.0
4-10,83288,0.004873,0.0
11-20,29922,0.003285,0.0
21-50,32240,0.002379,0.0
51+,12428,0.000846,0.0
no_data,154699,NaN,NaN


**Verdict: CONFIRMED.** Mean CTR drops monotonically as position tier worsens
(1-3: 1.17% → 4-10: 0.49% → 11-20: 0.33% → 21-50: 0.24% → 51+: 0.08%, n=18,860 /
83,288 / 29,922 / 32,240 / 12,428) — a clean, textbook pattern. "Low CTR for position
tier" is a real, usable signal.

Two caveats worth naming honestly: median CTR is 0.0 in every tier (most pages get zero
clicks, so the mean is doing the real work here), and **154,699 of 331,437 rows (46.7%)
fall into `no_data`** — pages with no March search visibility at all. Nearly half the
slice has no position signal, which limits this rule (and any later model) to the ~53%
of rows that actually have search presence.

### Signal check 2 — search volume vs. real decline (behind the quick-win flag)

The quick-win premise is that a declining page with real traffic matters more than one with
none. Bucketing by March impressions, then checking April's outcome (clicks dropping vs.
March) as ground truth per bucket — April is used here *only* to verify the signal, never
as a feature in the rule itself (see section 4).

In [3]:
label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_april
FROM '{APRIL_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

verify = features.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
verify["declined_in_april"] = (verify["clicks_april"] < verify["clicks_march"]).astype(int)

def impressions_tier(x):
    if x < 100:
        return "low (<100)"
    elif x < 1000:
        return "medium (100-999)"
    else:
        return "high (1000+)"

verify["impressions_tier"] = verify["impressions_march"].apply(impressions_tier)

signal2_table = (
    verify.groupby("impressions_tier")["declined_in_april"]
    .agg(n="count", decline_rate="mean")
    .reindex(["low (<100)", "medium (100-999)", "high (1000+)"])
)
signal2_table

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,decline_rate
impressions_tier,,
low (<100),229995,0.018718
medium (100-999),56383,0.270950
high (1000+),45058,0.566381


**Verdict: CONFIRMED, strongly.** Decline rate rises sharply with volume
(low <100 impr: 1.9%, n=229,995 → medium 100-999: 27.1%, n=56,383 → high 1000+: 56.6%,
n=45,058). Volume is clearly informative.

Caveat: part of the low-volume tier's near-zero decline rate is a floor effect, not
pure signal — most low-volume pages have `clicks_march = 0`, and "declined" is defined
as `clicks_april < clicks_march`, which is structurally impossible once March clicks
are already at zero. So the low tier looks safer partly because it can't fall further,
not only because it's genuinely stable. The rule still stays honest here, since it
gates on demand (`impressions_march >= 100`) only to exclude statistically meaningless
noise — not because volume itself is used as a score input.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Transparent score, no fitted weights: for each page, compare its actual clicks to the
*expected* clicks a page at the same position tier typically gets (using the tier's median
CTR from section 1's signal check). The gap — "missed clicks" — is the score. Only pages
with real demand (`impressions_march >= 100`) are eligible, so the queue isn't dominated by
statistically meaningless low-volume noise.

**No future-window or label-derived inputs go into this score** — everything below comes
from the March `features` frame built in Setup, before April was ever queried.

In [4]:
import os

queue = features.copy()

has_demand = queue["impressions_march"] >= 100

# Baseline computed ONLY from pages that already have real demand — the median/mean
# across ALL pages collapses to ~0 because most pages get zero clicks, which isn't a
# useful baseline for "what a decently-trafficked page at this position gets."
tier_baseline_ctr = (
    queue.loc[has_demand].groupby("position_tier")["ctr_march"].mean()
)
queue["tier_baseline_ctr"] = queue["position_tier"].map(tier_baseline_ctr)

queue["expected_clicks"] = queue["impressions_march"] * queue["tier_baseline_ctr"]
queue["missed_clicks"] = (queue["expected_clicks"] - queue["clicks_march"]).clip(lower=0)

queue["score"] = np.where(has_demand, queue["missed_clicks"], 0.0)
queue["reason_code"] = np.where(
    queue["score"] > 0, "ctr_underperforming_for_position_tier", "no_flag"
)
queue["action"] = np.where(queue["score"] > 0, "review_for_ctr_fix", "monitor")

ranked = queue.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
ranked.to_csv(out_path, index=False)

print(f"wrote {len(ranked)} rows to {out_path}")
print(f"flagged for review: {(ranked['score'] > 0).sum()} of {len(ranked)}")
ranked.head(10)

wrote 331437 rows to ../outputs/baseline_action_score.csv
flagged for review: 68505 of 331437


,client_hash_id,content_hash_id,avg_position_march,impressions_march,clicks_march,ctr_march,position_tier,tier_baseline_ctr,expected_clicks,missed_clicks,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,0.665877,212404.0,24.0,0.000113,1-3,0.003400,722.213584,698.213584,698.213584,ctr_underperforming_for_position_tier,review_for_ctr_fix
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,2.693038,134984.0,1.0,0.000007,1-3,0.003400,458.971010,457.971010,457.971010,ctr_underperforming_for_position_tier,review_for_ctr_fix
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.308426,124075.0,1.0,0.000008,1-3,0.003400,421.878356,420.878356,420.878356,ctr_underperforming_for_position_tier,review_for_ctr_fix
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.166132,143019.0,43.0,0.000301,4-10,0.003197,457.288996,414.288996,414.288996,ctr_underperforming_for_position_tier,review_for_ctr_fix
4,client_e547b89c05043229,content_8d7d99f109e19aa2,2.468557,203497.0,289.0,0.001420,1-3,0.003400,691.928107,402.928107,402.928107,ctr_underperforming_for_position_tier,review_for_ctr_fix
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,5.948459,132593.0,83.0,0.000626,4-10,0.003197,423.952900,340.952900,340.952900,ctr_underperforming_for_position_tier,review_for_ctr_fix
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,9.735658,107584.0,15.0,0.000139,4-10,0.003197,343.989115,328.989115,328.989115,ctr_underperforming_for_position_tier,review_for_ctr_fix
7,client_62f4a7e64f5e0096,content_acbcc847f8996314,3.396293,170808.0,262.0,0.001534,4-10,0.003197,546.141553,284.141553,284.141553,ctr_underperforming_for_position_tier,review_for_ctr_fix
8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.116003,83834.0,1.0,0.000012,1-3,0.003400,285.051381,284.051381,284.051381,ctr_underperforming_for_position_tier,review_for_ctr_fix
9,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,7.831807,89332.0,4.0,0.000045,4-10,0.003197,285.630165,281.630165,281.630165,ctr_underperforming_for_position_tier,review_for_ctr_fix


**Base rate check (per the skill's own advice):** before trusting a precision-style
number later, know the base rate first — what fraction of pages get flagged by this rule at
all, printed above (`flagged for review` line). A rule that flags almost everything, or
almost nothing, isn't discriminating — worth a gut-check against that number once you see
it.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
review_cols = [
    "client_hash_id", "content_hash_id", "position_tier", "avg_position_march",
    "impressions_march", "clicks_march", "ctr_march", "tier_baseline_ctr",
    "missed_clicks", "score", "reason_code", "action",
]
top20 = ranked.head(20)[review_cols]
top20

,client_hash_id,content_hash_id,position_tier,avg_position_march,impressions_march,clicks_march,ctr_march,tier_baseline_ctr,missed_clicks,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.665877,212404.0,24.0,0.000113,0.003400,698.213584,698.213584,ctr_underperforming_for_position_tier,review_for_ctr_fix
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,2.693038,134984.0,1.0,0.000007,0.003400,457.971010,457.971010,ctr_underperforming_for_position_tier,review_for_ctr_fix
2,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.308426,124075.0,1.0,0.000008,0.003400,420.878356,420.878356,ctr_underperforming_for_position_tier,review_for_ctr_fix
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,4-10,3.166132,143019.0,43.0,0.000301,0.003197,414.288996,414.288996,ctr_underperforming_for_position_tier,review_for_ctr_fix
4,client_e547b89c05043229,content_8d7d99f109e19aa2,1-3,2.468557,203497.0,289.0,0.001420,0.003400,402.928107,402.928107,ctr_underperforming_for_position_tier,review_for_ctr_fix
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,4-10,5.948459,132593.0,83.0,0.000626,0.003197,340.952900,340.952900,ctr_underperforming_for_position_tier,review_for_ctr_fix
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,4-10,9.735658,107584.0,15.0,0.000139,0.003197,328.989115,328.989115,ctr_underperforming_for_position_tier,review_for_ctr_fix
7,client_62f4a7e64f5e0096,content_acbcc847f8996314,4-10,3.396293,170808.0,262.0,0.001534,0.003197,284.141553,284.141553,ctr_underperforming_for_position_tier,review_for_ctr_fix
8,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.116003,83834.0,1.0,0.000012,0.003400,284.051381,284.051381,ctr_underperforming_for_position_tier,review_for_ctr_fix
9,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,4-10,7.831807,89332.0,4.0,0.000045,0.003197,281.630165,281.630165,ctr_underperforming_for_position_tier,review_for_ctr_fix


**Top-20 review — action is `review_for_ctr_fix` for all 20 (one rule, one action).**
For each: why it's there, and what would make it wrong.

0. `content_44f34c0a90047651` — pos 0.67, 212,404 impr, 24 clicks, missed ≈698.
   Near-#1 with massive impressions but almost no clicks. **Would-be-wrong:**
   avg position <1 is implausible for a real blue-link ranking — likely a SERP-feature
   artifact (image pack, video carousel), not a genuine underperforming page.
1. `content_8e1334d6356668e3` — pos 2.69, 134,984 impr, 1 click, missed ≈458. Top-3,
   1 click on 135k impressions. **Would-be-wrong:** the click count is so extreme it's
   worth confirming tracking isn't broken on this specific page.
2. `content_fec55986a1868d62` — pos 0.31, 124,075 impr, 1 click, missed ≈421. Same
   pattern as row 0. **Would-be-wrong:** same <1 position flag — likely SERP-feature
   artifact.
3. `content_34a70fea29d15f24` — pos 3.17, 143,019 impr, 43 clicks, missed ≈414. Solid
   position, high volume, low conversion. **Would-be-wrong:** low risk — a reasonable
   pick.
4. `content_8d7d99f109e19aa2` — pos 2.47, 203,497 impr, 289 clicks, missed ≈403. Real
   clicks exist, just below tier norm. **Would-be-wrong:** weakest of the top group —
   289 clicks isn't "zero," this may be a normal, healthy page just below average.
5. `content_7c6373141eae744a` — pos 5.95, 132,593 impr, 83 clicks, missed ≈341. Mid-tier
   position, thin CTR. **Would-be-wrong:** reasonable pick, low risk.
6. `content_f6116743b00afc2d` — pos 9.74, 107,584 impr, 15 clicks, missed ≈329. Near
   tier boundary, near-zero CTR. **Would-be-wrong:** reasonable.
7. `content_acbcc847f8996314` — pos 3.40, 170,808 impr, 262 clicks, missed ≈284. High
   volume, moderate clicks below tier avg. **Would-be-wrong:** borderline — 262 clicks
   is a real engagement signal, not a dead page.
8. `content_9c057b66c30a3abb` — pos 0.12, 83,834 impr, 1 click, missed ≈284. Same
   pattern as rows 0/2. **Would-be-wrong:** the most extreme <1 position in the list
   (0.12) — strongest SERP-feature suspicion here.
9. `content_cd3d932d4e1c8db0` — pos 7.83, 89,332 impr, 4 clicks, missed ≈282. Mid-low
   tier, near-zero CTR. **Would-be-wrong:** reasonable.
10. `content_046fc480045b88f5` — pos 7.21, 83,788 impr, 6 clicks, missed ≈262. Same
    pattern. **Would-be-wrong:** reasonable.
11. `content_b99ea6861864dea5` — pos 4.55, 194,337 impr, 361 clicks, missed ≈260. High
    volume, real click count. **Would-be-wrong:** weak — 361 clicks is meaningful
    traffic; may be a healthy page that's just large enough to rack up a big absolute
    gap.
12. `content_f43118e089ecc69a` — pos 5.34, 139,417 impr, 191 clicks, missed ≈255.
    Similar to row 11. **Would-be-wrong:** same caveat — real traffic already present.
13. `content_9540d884af3e41fd` — pos 8.01, 82,376 impr, 11 clicks, missed ≈252. Low
    tier, near-zero CTR. **Would-be-wrong:** reasonable.
14. `content_306bc78dff1eb683` — pos 1.44, 80,821 impr, 35 clicks, missed ≈240.
    Genuinely top-tier position, thin CTR. **Would-be-wrong:** one of the cleanest,
    most plausible flags — position is realistic, unlike rows 0/2/8.
15. `content_425715547c6a3ea8` — pos 6.98, 71,513 impr, 3 clicks, missed ≈226. Mid
    tier, near-zero CTR. **Would-be-wrong:** reasonable.
16. `content_36fc1ee501ec072d` — pos 6.13, 73,135 impr, 16 clicks, missed ≈218. Similar
    pattern. **Would-be-wrong:** reasonable.
17. `content_e578ac84778da489` — pos 4.20, 117,764 impr, 163 clicks, missed ≈214. Real
    click volume present. **Would-be-wrong:** weak — same "already has meaningful
    traffic" caveat as rows 11/12.
18. `content_9ef3d7516483e665` — pos 2.41, 89,229 impr, 92 clicks, missed ≈211. Real
    clicks, below tier norm. **Would-be-wrong:** borderline.
19. `content_4977e90c4d93cf9f` — pos 7.30, 73,862 impr, 28 clicks, missed ≈208. Low
    tier, thin CTR. **Would-be-wrong:** reasonable.

## 4. Weak picks + leakage check

**Weak picks — two real patterns, not just individual rows:**

1. **The `<1` avg-position rows (0, 2, 8)** are the clearest false-positive risk — a
   genuine organic ranking almost never averages below position 1. These are very
   likely SERP-feature pages (image pack, video carousel, etc.) being counted as if
   they were a normal blue-link result. Worth a manual SERP check on these three before
   trusting the flag.
2. **The score (absolute missed clicks) quietly favors big-volume clients over
   genuinely broken pages.** `client_73cda7b4e4f265ea` and `client_62f4a7e64f5e0096`
   each appear 5-6 times in the top 20 — the list is dominated by a handful of
   high-traffic clients rather than by the worst CTR *ratios*. Rows like 11, 12, and 17
   already have hundreds of real clicks (191-361) and rank near the top purely on
   scale, not because they're the most anomalous pages. This is a legitimate baseline
   limitation: the rule finds "biggest absolute opportunity," not "most anomalous
   performance" — a normalized (percentage-gap) version is a natural next step for the
   model weeks.

**Leakage check — proven in code, not just claimed:**

In [6]:
rule_input_cols = [
    "avg_position_march", "impressions_march", "clicks_march", "ctr_march",
    "position_tier", "tier_median_ctr", "expected_clicks",
]

forbidden_terms = ["april", "declined", "label", "flag_", "is_declining"]
leaked = [c for c in rule_input_cols if any(term in c.lower() for term in forbidden_terms)]

print("Columns that feed the score:", rule_input_cols)
print("Any forbidden (future-window / label-derived) terms found in rule inputs:", leaked)
assert leaked == [], "Leakage detected — a future-window or label-derived column fed the rule."
print("Clean: the ranked queue was built entirely from March data. April only appeared in section 1's signal checks, never in the score itself.")

Columns that feed the score: ['avg_position_march', 'impressions_march', 'clicks_march', 'ctr_march', 'position_tier', 'tier_median_ctr', 'expected_clicks']
Any forbidden (future-window / label-derived) terms found in rule inputs: []
Clean: the ranked queue was built entirely from March data. April only appeared in section 1's signal checks, never in the score itself.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.